In [8]:
import pandas as pd
import sys
sys.path.append('../src')
import warnings
warnings.filterwarnings('ignore')

from data_loader import construir_dataframe_maestro

df = construir_dataframe_maestro()
df_delivered = df[df['order_status'] == 'delivered'].copy()

print(f"DataFrame maestro: {df.shape[0]} filas, {df.shape[1]} columnas")

DataFrame maestro: 113425 filas, 34 columnas


In [4]:
# Solo pedidos entregados con review_score conocido
df_modelo = df_delivered.drop_duplicates(subset='order_id').copy()
df_modelo = df_modelo[df_modelo['review_score'].notna()]

# Target: 1 si review_score es 1 o 2, 0 si es 3,4,5
df_modelo['review_negativo'] = (df_modelo['review_score'] <= 2).astype(int)

# Feature de distancia logística (proxy simple)
df_modelo['mismo_estado'] = (df_modelo['customer_state'] == df_modelo['seller_state']).astype(int)

# Días estimados de entrega (permitido - se conoce al momento de la compra)
df_modelo['order_estimated_delivery_date'] = pd.to_datetime(df_modelo['order_estimated_delivery_date'])
df_modelo['order_purchase_timestamp'] = pd.to_datetime(df_modelo['order_purchase_timestamp'])
df_modelo['dias_estimados'] = (
    df_modelo['order_estimated_delivery_date'] - df_modelo['order_purchase_timestamp']
).dt.days

# Verificar balance de clases
print(df_modelo['review_negativo'].value_counts(normalize=True))
print(f"\nTotal de filas para modelado: {len(df_modelo)}")

review_negativo
0    0.871995
1    0.128005
Name: proportion, dtype: float64

Total de filas para modelado: 95832


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Selecciona features (sin data leakage)
features_numericas = ['price', 'freight_value', 'payment_installments_max', 
                       'dias_estimados', 'mismo_estado']
features_categoricas = ['payment_type_principal', 'product_category_name_english']

# Elimina filas con nulos en las features seleccionadas
df_modelo_limpio = df_modelo.dropna(subset=features_numericas + features_categoricas + ['review_negativo'])

print(f"Filas después de eliminar nulos en features: {len(df_modelo_limpio)}")
print(f"Filas eliminadas: {len(df_modelo) - len(df_modelo_limpio)}")

# One-hot encoding para categóricas
df_encoded = pd.get_dummies(df_modelo_limpio[features_numericas + features_categoricas], 
                              columns=features_categoricas, drop_first=True)

X = df_encoded
y = df_modelo_limpio['review_negativo']

# Split estratificado - IMPORTANTE con clases desbalanceadas, para que 
# ambos conjuntos (train/test) mantengan la misma proporción 87/13
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape[0]} filas, {X_train.shape[1]} columnas")
print(f"Test: {X_test.shape[0]} filas")
print(f"Proporción clase 1 en train: {y_train.mean():.3f}")
print(f"Proporción clase 1 en test: {y_test.mean():.3f}")

Filas después de eliminar nulos en features: 94462
Filas eliminadas: 1370

Train: 75569 filas, 78 columnas
Test: 18893 filas
Proporción clase 1 en train: 0.127
Proporción clase 1 en test: 0.127


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# class_weight='balanced' compensa el desbalance ajustando internamente 
# la importancia de cada clase durante el entrenamiento
modelo_baseline = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
modelo_baseline.fit(X_train, y_train)

y_pred = modelo_baseline.predict(X_test)
y_pred_proba = modelo_baseline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Satisfecho', 'Insatisfecho']))
print(f"\nAUC-ROC: {roc_auc_score(y_test, y_pred_proba):.3f}")

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

  Satisfecho       0.89      0.58      0.70     16486
Insatisfecho       0.16      0.53      0.24      2407

    accuracy                           0.57     18893
   macro avg       0.53      0.56      0.47     18893
weighted avg       0.80      0.57      0.65     18893


AUC-ROC: 0.576

Matriz de confusión:
[[9586 6900]
 [1133 1274]]


In [11]:
from sklearn.ensemble import RandomForestClassifier

modelo_rf = RandomForestClassifier(
    n_estimators=200, 
    class_weight='balanced', 
    max_depth=10,
    random_state=42
)
modelo_rf.fit(X_train, y_train)

y_pred_rf = modelo_rf.predict(X_test)
y_pred_proba_rf = modelo_rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf, target_names=['Satisfecho', 'Insatisfecho']))
print(f"\nAUC-ROC: {roc_auc_score(y_test, y_pred_proba_rf):.3f}")

              precision    recall  f1-score   support

  Satisfecho       0.90      0.54      0.67     16486
Insatisfecho       0.15      0.57      0.24      2407

    accuracy                           0.54     18893
   macro avg       0.52      0.55      0.46     18893
weighted avg       0.80      0.54      0.62     18893


AUC-ROC: 0.581


In [12]:
importancias = pd.DataFrame({
    'feature': X_train.columns,
    'importancia': modelo_rf.feature_importances_
}).sort_values('importancia', ascending=False)

print(importancias.head(15))

                                              feature  importancia
1                                       freight_value     0.234098
3                                      dias_estimados     0.192848
0                                               price     0.181953
2                            payment_installments_max     0.090837
4                                        mismo_estado     0.077899
14       product_category_name_english_bed_bath_table     0.030433
64     product_category_name_english_office_furniture     0.024924
5                  payment_type_principal_credit_card     0.015085
7                      payment_type_principal_voucher     0.010194
46      product_category_name_english_furniture_decor     0.009305
73           product_category_name_english_stationery     0.008740
15  product_category_name_english_books_general_in...     0.008563
6                   payment_type_principal_debit_card     0.008307
60  product_category_name_english_luggage_accessories     0.00

In [13]:
# Peso y dimensiones del producto (afecta manejo/riesgo de daño en tránsito)
df_modelo['volumen_cm3'] = (
    df_modelo['product_length_cm'] * df_modelo['product_height_cm'] * df_modelo['product_width_cm']
)

# Día de la semana y si es fin de semana (podría afectar tiempos de procesamiento)
df_modelo['dia_semana_compra'] = df_modelo['order_purchase_timestamp'].dt.dayofweek
df_modelo['es_fin_de_semana'] = (df_modelo['dia_semana_compra'] >= 5).astype(int)

# Ratio flete/precio - un flete caro relativo al producto podría generar percepción negativa
df_modelo['ratio_flete_precio'] = df_modelo['freight_value'] / df_modelo['price']

In [14]:
# Features nuevos (sin data leakage)
df_modelo['volumen_cm3'] = (
    df_modelo['product_length_cm'] * df_modelo['product_height_cm'] * df_modelo['product_width_cm']
)
df_modelo['dia_semana_compra'] = df_modelo['order_purchase_timestamp'].dt.dayofweek
df_modelo['es_fin_de_semana'] = (df_modelo['dia_semana_compra'] >= 5).astype(int)
df_modelo['ratio_flete_precio'] = df_modelo['freight_value'] / df_modelo['price']

# Actualiza la lista de features numéricas
features_numericas_v2 = ['price', 'freight_value', 'payment_installments_max', 
                          'dias_estimados', 'mismo_estado', 'volumen_cm3',
                          'es_fin_de_semana', 'ratio_flete_precio']

df_modelo_limpio_v2 = df_modelo.dropna(subset=features_numericas_v2 + features_categoricas + ['review_negativo'])

print(f"Filas después de eliminar nulos: {len(df_modelo_limpio_v2)}")

df_encoded_v2 = pd.get_dummies(df_modelo_limpio_v2[features_numericas_v2 + features_categoricas], 
                                 columns=features_categoricas, drop_first=True)

X_v2 = df_encoded_v2
y_v2 = df_modelo_limpio_v2['review_negativo']

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2, y_v2, test_size=0.2, random_state=42, stratify=y_v2
)

modelo_rf_v2 = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', max_depth=10, random_state=42
)
modelo_rf_v2.fit(X_train_v2, y_train_v2)

y_pred_proba_v2 = modelo_rf_v2.predict_proba(X_test_v2)[:, 1]
print(f"\nAUC-ROC nuevo: {roc_auc_score(y_test_v2, y_pred_proba_v2):.3f}")
print(f"AUC-ROC anterior: 0.581")

Filas después de eliminar nulos: 94462

AUC-ROC nuevo: 0.585
AUC-ROC anterior: 0.581


## Conclusión: Límite del enfoque predictivo con datos pre-compra

Se probaron dos algoritmos (regresión logística, Random Forest) y dos 
iteraciones de features (8 variables base, luego +4 features derivados: 
volumen del producto, día de la semana, ratio flete/precio). El AUC-ROC 
se mantuvo estable entre 0.576 y 0.585 en todos los casos, indicando 
que el techo no es el algoritmo sino la información disponible.

**Interpretación de negocio**: la satisfacción del cliente parece depender 
principalmente de factores que ocurren DESPUÉS de la compra (tiempo real 
de entrega, calidad del empaque, estado del producto al llegar) — 
confirmando el hallazgo del EDA (correlación de -0.341 entre días de 
entrega reales y calificación). Esto sugiere que un sistema de alerta 
temprana necesitaría datos de seguimiento del pedido en tránsito, no solo 
datos del momento de la compra.

**Recomendación de negocio alternativa**: en vez de predecir riesgo al 
momento de compra, sería más efectivo un modelo de re-scoring dinámico 
que actualice el riesgo conforme avanza el envío (ej. si un pedido ya 
lleva más días de los estimados sin moverse de status).

In [15]:
# Traer las columnas de fecha que no usamos en el modelo anterior
df_modelo['order_approved_at'] = pd.to_datetime(df_modelo['order_approved_at'])
df_modelo['order_delivered_carrier_date'] = pd.to_datetime(df_modelo['order_delivered_carrier_date'])

# Días que tardó en despacharse desde la compra
df_modelo['dias_hasta_envio'] = (
    df_modelo['order_delivered_carrier_date'] - df_modelo['order_purchase_timestamp']
).dt.days

# El feature clave: cuánto margen queda respecto a lo estimado, AL MOMENTO DEL ENVÍO
# Si es negativo, el pedido YA salió tarde según lo prometido al cliente
df_modelo['margen_al_envio'] = df_modelo['dias_estimados'] - df_modelo['dias_hasta_envio']

# Verifica cuántos pedidos tienen este dato disponible
print(f"Pedidos con fecha de envío a transportista: {df_modelo['order_delivered_carrier_date'].notna().sum()}")
print(f"Total de pedidos en df_modelo: {len(df_modelo)}")
print(f"\nEstadísticas de margen_al_envio:")
print(df_modelo['margen_al_envio'].describe())

Pedidos con fecha de envío a transportista: 95830
Total de pedidos en df_modelo: 95832

Estadísticas de margen_al_envio:
count    95830.000000
mean        20.638234
std          8.759021
min        -99.000000
25%         15.000000
50%         20.000000
75%         26.000000
max        193.000000
Name: margen_al_envio, dtype: float64


In [16]:
# Investiga los casos más extremos en ambas direcciones
print("Los 5 casos con MAYOR atraso al momento de envío (margen más negativo):")
print(df_modelo.nsmallest(5, 'margen_al_envio')[
    ['order_id', 'order_purchase_timestamp', 'order_estimated_delivery_date', 
     'order_delivered_carrier_date', 'dias_estimados', 'dias_hasta_envio', 'margen_al_envio']
])

print("\nLos 5 casos con MAYOR margen (envío muy anticipado):")
print(df_modelo.nlargest(5, 'margen_al_envio')[
    ['order_id', 'order_purchase_timestamp', 'order_estimated_delivery_date', 
     'order_delivered_carrier_date', 'dias_estimados', 'dias_hasta_envio', 'margen_al_envio']
])

Los 5 casos con MAYOR atraso al momento de envío (margen más negativo):
                               order_id order_purchase_timestamp  \
41381  da81fbc27b55e0f3d2813cf2078dc780      2017-11-14 21:07:55   
3462   8b7fd198ad184563c231653673e75a7f      2017-11-14 10:04:27   
40136  97f48024fcc76f1898e397ad6966e3a0      2017-11-29 12:25:00   
50370  866314550f6d7a55c82917d9b4463e1f      2017-11-16 14:55:04   
35037  bfbd0f9bdef84302105ad712db648a6c      2016-09-15 12:16:38   

      order_estimated_delivery_date order_delivered_carrier_date  \
41381                    2017-12-11          2018-03-20 15:44:40   
3462                     2017-11-28          2018-02-26 17:27:15   
40136                    2017-12-26          2018-03-16 13:58:02   
50370                    2017-12-13          2018-01-21 16:12:17   
35037                    2016-10-04          2016-11-07 17:11:53   

       dias_estimados  dias_hasta_envio  margen_al_envio  
41381              26             125.0            

In [17]:
# ¿Cuántos casos tienen la imposibilidad lógica: envío antes de la compra?
envios_imposibles = df_modelo[df_modelo['dias_hasta_envio'] < 0]
print(f"Pedidos con fecha de envío ANTES de la compra (imposible): {len(envios_imposibles)}")

# ¿Cuántos tienen dias_estimados sospechosamente altos?
print(f"\nPedidos con dias_estimados > 60 (revisar si son reales o error):")
print(df_modelo[df_modelo['dias_estimados'] > 60]['dias_estimados'].describe())

Pedidos con fecha de envío ANTES de la compra (imposible): 164

Pedidos con dias_estimados > 60 (revisar si son reales o error):
count    212.000000
mean      70.693396
std       14.607839
min       61.000000
25%       62.750000
50%       66.000000
75%       72.000000
max      155.000000
Name: dias_estimados, dtype: float64


In [18]:
# ¿Hay traslape entre los 164 imposibles y los 212 de estimado alto?
casos_estimado_alto = df_modelo[df_modelo['dias_estimados'] > 60]
traslape = casos_estimado_alto[casos_estimado_alto['dias_hasta_envio'] < 0]
print(f"Casos con estimado alto QUE TAMBIÉN son imposibles: {len(traslape)} de {len(casos_estimado_alto)}")

# ¿Dónde están geográficamente los casos de estimado alto (excluyendo los imposibles)?
casos_estimado_alto_validos = casos_estimado_alto[casos_estimado_alto['dias_hasta_envio'] >= 0]
print(f"\nDistribución geográfica de estimados altos (válidos, {len(casos_estimado_alto_validos)} casos):")
print(casos_estimado_alto_validos['customer_state'].value_counts().head(10))

Casos con estimado alto QUE TAMBIÉN son imposibles: 0 de 212

Distribución geográfica de estimados altos (válidos, 212 casos):
customer_state
RJ    31
SP    28
MG    18
RS    15
PE    12
PA    12
BA    10
PR     9
MA     9
CE     8
Name: count, dtype: int64


In [19]:
# Excluir: envíos imposibles (antes de compra) + estimados sospechosamente altos
df_modelo_v3 = df_modelo[
    (df_modelo['dias_hasta_envio'] >= 0) & 
    (df_modelo['dias_estimados'] <= 60)
].copy()

print(f"Dataset original: {len(df_modelo)} filas")
print(f"Dataset limpio para modelo de re-scoring: {len(df_modelo_v3)} filas")
print(f"Filas excluidas: {len(df_modelo) - len(df_modelo_v3)}")

# Verifica que margen_al_envio ya no tenga outliers imposibles
print(f"\nNuevo rango de margen_al_envio:")
print(df_modelo_v3['margen_al_envio'].describe())

Dataset original: 95832 filas
Dataset limpio para modelo de re-scoring: 95454 filas
Filas excluidas: 378

Nuevo rango de margen_al_envio:
count    95454.000000
mean        20.548149
std          8.514575
min        -99.000000
25%         15.000000
50%         20.000000
75%         26.000000
max         60.000000
Name: margen_al_envio, dtype: float64


## Limpieza para modelo de re-scoring (checkpoint de envío)

Se identificaron y excluyeron dos tipos de datos problemáticos:
- 164 pedidos (0.17%) con fecha de envío al transportista ANTERIOR a la 
  fecha de compra — imposibilidad lógica, error de datos.
- 212 pedidos (0.22%) con `dias_estimados` > 60 días, concentrados 
  principalmente en RJ y SP (estados con mejor infraestructura logística 
  del país), descartando la hipótesis de que fueran casos legítimos de 
  distancia geográfica real.

Ambos grupos se excluyen del dataset de modelado por no ser representativos 
de la operación normal.

In [20]:
features_checkpoint = ['price', 'freight_value', 'payment_installments_max',
                        'dias_estimados', 'mismo_estado', 'ratio_flete_precio',
                        'dias_hasta_envio', 'margen_al_envio']

df_final = df_modelo_v3.dropna(subset=features_checkpoint + features_categoricas + ['review_negativo'])

print(f"Filas para el modelo final: {len(df_final)}")

df_encoded_final = pd.get_dummies(
    df_final[features_checkpoint + features_categoricas],
    columns=features_categoricas, drop_first=True
)

X_final = df_encoded_final
y_final = df_final['review_negativo']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42, stratify=y_final
)

modelo_checkpoint = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', max_depth=10, random_state=42
)
modelo_checkpoint.fit(X_train_f, y_train_f)

y_pred_proba_checkpoint = modelo_checkpoint.predict_proba(X_test_f)[:, 1]
auc_checkpoint = roc_auc_score(y_test_f, y_pred_proba_checkpoint)

print(f"AUC-ROC modelo checkpoint de envío: {auc_checkpoint:.3f}")
print(f"AUC-ROC modelo original (solo compra): 0.585")

Filas para el modelo final: 94086
AUC-ROC modelo checkpoint de envío: 0.620
AUC-ROC modelo original (solo compra): 0.585


In [21]:
y_pred_checkpoint = modelo_checkpoint.predict(X_test_f)
print(classification_report(y_test_f, y_pred_checkpoint, target_names=['Satisfecho', 'Insatisfecho']))

# Importancia de features - ¿el checkpoint de envío realmente domina?
importancias_v2 = pd.DataFrame({
    'feature': X_train_f.columns,
    'importancia': modelo_checkpoint.feature_importances_
}).sort_values('importancia', ascending=False)

print("\nTop 10 features más importantes:")
print(importancias_v2.head(10))

              precision    recall  f1-score   support

  Satisfecho       0.90      0.71      0.79     16420
Insatisfecho       0.19      0.45      0.26      2398

    accuracy                           0.68     18818
   macro avg       0.54      0.58      0.53     18818
weighted avg       0.81      0.68      0.73     18818


Top 10 features más importantes:
                                         feature  importancia
6                               dias_hasta_envio     0.237066
7                                margen_al_envio     0.169509
1                                  freight_value     0.110594
3                                 dias_estimados     0.087067
5                             ratio_flete_precio     0.079562
0                                          price     0.075240
4                                   mismo_estado     0.057833
2                       payment_installments_max     0.046267
17  product_category_name_english_bed_bath_table     0.015769
8             payme

In [22]:
from sklearn.metrics import precision_recall_curve

precision, recall, umbrales = precision_recall_curve(y_test_f, y_pred_proba_checkpoint)

# Encuentra el umbral que da al menos 0.65 de recall, maximizando precision
import numpy as np
idx_recall_65 = np.argmin(np.abs(recall - 0.65))
umbral_optimo = umbrales[idx_recall_65] if idx_recall_65 < len(umbrales) else 0.5

print(f"Umbral sugerido para recall ~0.65: {umbral_optimo:.3f}")

y_pred_ajustado = (y_pred_proba_checkpoint >= umbral_optimo).astype(int)
print("\nCon umbral ajustado:")
print(classification_report(y_test_f, y_pred_ajustado, target_names=['Satisfecho', 'Insatisfecho']))

Umbral sugerido para recall ~0.65: 0.479

Con umbral ajustado:
              precision    recall  f1-score   support

  Satisfecho       0.91      0.51      0.65     16420
Insatisfecho       0.16      0.65      0.26      2398

    accuracy                           0.53     18818
   macro avg       0.54      0.58      0.46     18818
weighted avg       0.81      0.53      0.60     18818



## Conclusión: Modelo de re-scoring en checkpoint de envío

Se construyó un modelo de Random Forest que evalúa el riesgo de 
insatisfacción del cliente en el momento en que el pedido se despacha 
al transportista (no al momento de la compra), incorporando `dias_hasta_envio` 
y `margen_al_envio` como features clave.

**Resultado**: AUC-ROC de 0.620, una mejora real sobre el modelo de solo-compra 
(0.585) — modesta pero consistente con el hallazgo del EDA de que el tiempo 
de manejo logístico es el factor más influyente en satisfacción.

**Trade-off precision/recall**: con umbral por defecto (0.5), recall=0.45, 
precision=0.19. Ajustando el umbral a 0.479 para priorizar detección 
(recall=0.65), la precision cae a 0.16 — de cada 100 alertas, solo 16 
serían casos reales de riesgo.

**Recomendación de negocio**: el modelo es útil como señal de apoyo para 
priorización (ej. "revisar primero los pedidos con mayor score de riesgo" 
en un equipo de atención al cliente con capacidad limitada), pero NO es 
confiable para automatizar decisiones sin supervisión humana, dado el 
alto nivel de falsos positivos en cualquier umbral razonable.

**Limitación honesta**: ningún modelo probado en este proyecto (regresión 
logística, Random Forest con 2 iteraciones de features, y este checkpoint 
de envío) superó AUC=0.62. Esto sugiere que factores no capturados en 
este dataset — calidad real del producto, comunicación del vendedor, 
expectativas subjetivas del cliente — probablemente tienen más peso en 
la satisfacción que las variables logísticas/transaccionales disponibles.

In [23]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

# Guarda el modelo y las columnas exactas que espera (para replicar el encoding en producción)
joblib.dump(modelo_checkpoint, '../models/modelo_riesgo_insatisfaccion.pkl')
joblib.dump(list(X_train_f.columns), '../models/columnas_modelo.pkl')

print("Modelo guardado correctamente")
print(f"Número de features esperadas: {len(X_train_f.columns)}")

Modelo guardado correctamente
Número de features esperadas: 81
